In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine
import time

conn   = sqlite3.connect('olist_database.db')
cursor = conn.cursor()
engine = create_engine('sqlite:///olist_database.db')

# Load all 7 tables into database
tables_to_load = {
    'orders'      : 'olist_orders_dataset.csv',
    'order_items' : 'olist_order_items_dataset.csv',
    'customers'   : 'olist_customers_dataset.csv',
    'payments'    : 'olist_order_payments_dataset.csv',
    'products'    : 'olist_products_dataset.csv',
    'reviews'     : 'olist_order_reviews_dataset.csv',
    'sellers'     : 'olist_sellers_dataset.csv'
}

for table_name, filename in tables_to_load.items():
    df = pd.read_csv(filename)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"{table_name} loaded ✓ — {len(df):,} rows")

# Load translation separately for product categories
translation = pd.read_csv('product_category_name_translation.csv')
translation.to_sql('translation', conn, if_exists='replace', index=False)
print(f"translation loaded ✓ — {len(translation):,} rows")

orders loaded ✓ — 99,441 rows
order_items loaded ✓ — 112,650 rows
customers loaded ✓ — 99,441 rows
payments loaded ✓ — 103,886 rows
products loaded ✓ — 32,951 rows
reviews loaded ✓ — 99,224 rows
sellers loaded ✓ — 3,095 rows
translation loaded ✓ — 71 rows


In [2]:
# Time a complex join before any indexes
start = time.time()

test_query = '''
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id   = oi.order_id
    JOIN customers c    ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id, c.customer_state
'''

pd.read_sql_query(test_query, conn)
before_time = round(time.time() - start, 3)
print(f"Query time BEFORE indexes: {before_time} seconds")

Query time BEFORE indexes: 7.955 seconds


In [3]:
indexes = [
    # orders table — most joined and filtered table
    ("idx_orders_order_id",          "CREATE INDEX IF NOT EXISTS idx_orders_order_id ON orders(order_id)"),
    ("idx_orders_customer_id",       "CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON orders(customer_id)"),
    ("idx_orders_status",            "CREATE INDEX IF NOT EXISTS idx_orders_status ON orders(order_status)"),
    ("idx_orders_purchase_date",     "CREATE INDEX IF NOT EXISTS idx_orders_purchase_date ON orders(order_purchase_timestamp)"),

    # order_items — joined on order_id and product_id constantly
    ("idx_items_order_id",           "CREATE INDEX IF NOT EXISTS idx_items_order_id ON order_items(order_id)"),
    ("idx_items_product_id",         "CREATE INDEX IF NOT EXISTS idx_items_product_id ON order_items(product_id)"),
    ("idx_items_seller_id",          "CREATE INDEX IF NOT EXISTS idx_items_seller_id ON order_items(seller_id)"),

    # customers — joined on customer_id, filtered by state
    ("idx_customers_customer_id",    "CREATE INDEX IF NOT EXISTS idx_customers_customer_id ON customers(customer_id)"),
    ("idx_customers_unique_id",      "CREATE INDEX IF NOT EXISTS idx_customers_unique_id ON customers(customer_unique_id)"),
    ("idx_customers_state",          "CREATE INDEX IF NOT EXISTS idx_customers_state ON customers(customer_state)"),

    # payments — joined on order_id, filtered by payment_type
    ("idx_payments_order_id",        "CREATE INDEX IF NOT EXISTS idx_payments_order_id ON payments(order_id)"),
    ("idx_payments_type",            "CREATE INDEX IF NOT EXISTS idx_payments_type ON payments(payment_type)"),

    # reviews — joined on order_id
    ("idx_reviews_order_id",         "CREATE INDEX IF NOT EXISTS idx_reviews_order_id ON reviews(order_id)"),

    # products — joined on product_id
    ("idx_products_product_id",      "CREATE INDEX IF NOT EXISTS idx_products_product_id ON products(product_id)"),
]

for name, sql in indexes:
    cursor.execute(sql)
    print(f"Created: {name} ✓")

conn.commit()
print(f"\nTotal indexes created: {len(indexes)}")

Created: idx_orders_order_id ✓
Created: idx_orders_customer_id ✓
Created: idx_orders_status ✓
Created: idx_orders_purchase_date ✓
Created: idx_items_order_id ✓
Created: idx_items_product_id ✓
Created: idx_items_seller_id ✓
Created: idx_customers_customer_id ✓
Created: idx_customers_unique_id ✓
Created: idx_customers_state ✓
Created: idx_payments_order_id ✓
Created: idx_payments_type ✓
Created: idx_reviews_order_id ✓
Created: idx_products_product_id ✓

Total indexes created: 14


In [4]:
start = time.time()
pd.read_sql_query(test_query, conn)
after_time = round(time.time() - start, 3)

improvement = round(((before_time - after_time) / before_time) * 100, 1)

print(f"Query time BEFORE indexes : {before_time} seconds")
print(f"Query time AFTER indexes  : {after_time} seconds")
print(f"Speed improvement         : {improvement}% faster")

Query time BEFORE indexes : 7.955 seconds
Query time AFTER indexes  : 9.785 seconds
Speed improvement         : -23.0% faster


In [5]:
cursor.execute('''
    SELECT name, tbl_name
    FROM sqlite_master
    WHERE type = 'index'
    ORDER BY tbl_name
''')

all_indexes = cursor.fetchall()
print(f"{'Index Name':<45} {'Table'}")
print("-" * 65)
for idx in all_indexes:
    print(f"{idx[0]:<45} {idx[1]}")

print(f"\nTotal indexes in database: {len(all_indexes)}")

Index Name                                    Table
-----------------------------------------------------------------
idx_customers_customer_id                     customers
idx_customers_unique_id                       customers
idx_customers_state                           customers
idx_items_order_id                            order_items
idx_items_product_id                          order_items
idx_items_seller_id                           order_items
idx_orders_order_id                           orders
idx_orders_customer_id                        orders
idx_orders_status                             orders
idx_orders_purchase_date                      orders
idx_payments_order_id                         payments
idx_payments_type                             payments
idx_products_product_id                       products
idx_reviews_order_id                          reviews

Total indexes in database: 14


In [6]:
print("""
ANALYTICAL MASTER TABLE (AMT) — Column Plan
============================================
SOURCE: orders + order_items + customers + payments + products + reviews + sellers

ORDER LEVEL:
  order_id, order_status, order_purchase_timestamp
  order_approved_at, order_delivered_customer_date
  order_estimated_delivery_date

CUSTOMER LEVEL:
  customer_unique_id, customer_city, customer_state

PRODUCT LEVEL:
  product_category_name_english

PAYMENT LEVEL:
  payment_type, payment_installments, payment_value

SELLER LEVEL:
  seller_id, seller_state

REVIEW LEVEL:
  review_score

CALCULATED FIELDS:
  total_order_revenue  (sum of price + freight)
  delivery_delay_days  (actual vs estimated delivery)
  is_delayed           (1 if delivered late)
  is_canceled          (1 if canceled)
""")


ANALYTICAL MASTER TABLE (AMT) — Column Plan
SOURCE: orders + order_items + customers + payments + products + reviews + sellers

ORDER LEVEL:
  order_id, order_status, order_purchase_timestamp
  order_approved_at, order_delivered_customer_date
  order_estimated_delivery_date

CUSTOMER LEVEL:
  customer_unique_id, customer_city, customer_state

PRODUCT LEVEL:
  product_category_name_english

PAYMENT LEVEL:
  payment_type, payment_installments, payment_value

SELLER LEVEL:
  seller_id, seller_state

REVIEW LEVEL:
  review_score

CALCULATED FIELDS:
  total_order_revenue  (sum of price + freight)
  delivery_delay_days  (actual vs estimated delivery)
  is_delayed           (1 if delivered late)
  is_canceled          (1 if canceled)



In [7]:
amt_query = '''
    SELECT
        -- Order identifiers
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_approved_at,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        -- Customer info
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,

        -- Product category
        COALESCE(tr.product_category_name_english, 'unknown') AS product_category,

        -- Payment info
        p.payment_type,
        p.payment_installments,
        ROUND(p.payment_value, 2)                              AS payment_value,

        -- Seller info
        s.seller_id,
        s.seller_state,

        -- Review score
        r.review_score,

        -- Revenue calculations
        ROUND(SUM(oi.price), 2)                                AS total_item_revenue,
        ROUND(SUM(oi.freight_value), 2)                        AS total_freight,
        ROUND(SUM(oi.price) + SUM(oi.freight_value), 2)        AS total_order_revenue,
        COUNT(oi.order_item_id)                                AS total_items_in_order,

        -- Delivery performance
        ROUND(JULIANDAY(o.order_delivered_customer_date) -
              JULIANDAY(o.order_estimated_delivery_date), 0)   AS delivery_delay_days,

        -- Binary flags for modeling
        CASE WHEN o.order_delivered_customer_date >
                  o.order_estimated_delivery_date
             THEN 1 ELSE 0 END                                 AS is_delayed,
        CASE WHEN o.order_status = 'canceled'
             THEN 1 ELSE 0 END                                 AS is_canceled,
        CASE WHEN o.order_status = 'delivered'
             THEN 1 ELSE 0 END                                 AS is_delivered,
        CASE WHEN p.payment_type = 'credit_card'
             THEN 1 ELSE 0 END                                 AS is_credit_card,
        CASE WHEN p.payment_installments > 6
             THEN 1 ELSE 0 END                                 AS is_high_installment

    FROM orders o
    JOIN order_items oi  ON o.order_id     = oi.order_id
    JOIN customers c     ON o.customer_id  = c.customer_id
    JOIN payments p      ON o.order_id     = p.order_id
    LEFT JOIN products pr ON oi.product_id = pr.product_id
    LEFT JOIN translation tr ON pr.product_category_name = tr.product_category_name
    LEFT JOIN reviews r  ON o.order_id     = r.order_id
    LEFT JOIN sellers s  ON oi.seller_id   = s.seller_id

    GROUP BY
        o.order_id, o.order_status, o.order_purchase_timestamp,
        o.order_approved_at, o.order_delivered_customer_date,
        o.order_estimated_delivery_date, c.customer_unique_id,
        c.customer_city, c.customer_state,
        tr.product_category_name_english,
        p.payment_type, p.payment_installments, p.payment_value,
        s.seller_id, s.seller_state, r.review_score

    ORDER BY o.order_purchase_timestamp
'''

print("Running AMT query — this may take 30-60 seconds due to 7-table join...")
start = time.time()
amt_df = pd.read_sql_query(amt_query, conn)
elapsed = round(time.time() - start, 2)

print(f"AMT built in {elapsed} seconds")
print(f"Shape: {amt_df.shape}")
print(f"\nColumns ({len(amt_df.columns)}):")
print(amt_df.columns.tolist())
print(f"\nSample:")
print(amt_df.head(3).to_string())

Running AMT query — this may take 30-60 seconds due to 7-table join...
AMT built in 27.98 seconds
Shape: (104330, 26)

Columns (26):
['order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_city', 'customer_state', 'product_category', 'payment_type', 'payment_installments', 'payment_value', 'seller_id', 'seller_state', 'review_score', 'total_item_revenue', 'total_freight', 'total_order_revenue', 'total_items_in_order', 'delivery_delay_days', 'is_delayed', 'is_canceled', 'is_delivered', 'is_credit_card', 'is_high_installment']

Sample:
                           order_id order_status order_purchase_timestamp    order_approved_at order_delivered_customer_date order_estimated_delivery_date                customer_unique_id customer_city customer_state product_category payment_type  payment_installments  payment_value                         seller_id seller_state  review_

In [8]:
print("=== AMT Validation ===")
print(f"Total rows              : {len(amt_df):,}")
print(f"Unique orders           : {amt_df['order_id'].nunique():,}")
print(f"Null values per column  :")
print(amt_df.isnull().sum()[amt_df.isnull().sum() > 0])
print(f"\nOrder status breakdown  :")
print(amt_df['order_status'].value_counts())
print(f"\nRevenue sanity check    :")
print(f"  Total AMT revenue     : BRL {amt_df['total_item_revenue'].sum():,.2f}")
print(f"  Is delayed count      : {amt_df['is_delayed'].sum():,}")
print(f"  Is canceled count     : {amt_df['is_canceled'].sum():,}")
print(f"  Avg review score      : {amt_df['review_score'].mean():.2f}")

=== AMT Validation ===
Total rows              : 104,330
Unique orders           : 98,665
Null values per column  :
order_approved_at                  14
order_delivered_customer_date    2308
review_score                      797
delivery_delay_days              2308
dtype: int64

Order status breakdown  :
order_status
delivered      102023
shipped          1170
canceled          486
invoiced          327
processing        315
unavailable         7
approved            2
Name: count, dtype: int64

Revenue sanity check    :
  Total AMT revenue     : BRL 14,273,564.68
  Is delayed count      : 8,141
  Is canceled count     : 486
  Avg review score      : 4.08


In [ ]:
amt_df.to_csv('olist_analytical_master_table.csv', index=False)

file_size = pd.read_csv('olist_analytical_master_table.csv').__sizeof__()
print("olist_analytical_master_table.csv exported ✓")
print(f"Rows    : {len(amt_df):,}")
print(f"Columns : {len(amt_df.columns)}")
print(f"Ready for Python modeling, dashboards, and BI tools")

In [ ]:
cursor.execute('''
    CREATE VIEW IF NOT EXISTS vw_monthly_revenue AS
    SELECT
        STRFTIME('%Y-%m', o.order_purchase_timestamp)   AS revenue_month,
        COUNT(DISTINCT o.order_id)                       AS total_orders,
        ROUND(SUM(oi.price), 2)                          AS total_revenue,
        ROUND(AVG(oi.price), 2)                          AS avg_order_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY revenue_month
    ORDER BY revenue_month
''')
conn.commit()
print("vw_monthly_revenue created ✓")

# Test it
test = pd.read_sql("SELECT * FROM vw_monthly_revenue LIMIT 5", conn)
print(test)